# Clusterização part 1

**Status:** Versão de referência · **Área:** ideb-regiao-norte

Versão inicial de seleção/deltas/KMeans/radar na região Norte.

Este arquivo está identificado como **Versão de referência**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


SUBIR O ARQUIVO A SER VISUALIZADO - ZIPADO

In [ ]:
import zipfile
def descompactar_arquivos(nome_arquivo_zip, destino):
  """
  Descompacta um arquivo zip.

  Args:
    divulgacao_anos_finais_municipios_2023(1).zip: O nome do arquivo zip a ser descompactado.
    destino: O diretório onde os arquivos serão extraídos.
  """
  try:
    with zipfile.ZipFile(nome_arquivo_zip, 'r') as zip_ref:
      zip_ref.extractall(destino)
    print(f"Arquivo {nome_arquivo_zip} descompactado com sucesso para {destino}")
  except FileNotFoundError:
    print(f"Arquivo {nome_arquivo_zip} não encontrado.")
  except Exception as e:
    print(f"Erro ao descompactar o arquivo: {e}")

In [ ]:
import pandas as pd

df = pd.read_excel('/content/planilhaIDEB.xlsx', skiprows=9)
df.head()

In [ ]:
print(df.columns)

In [ ]:
regiao_norte = ['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO']
df = df[df['SG_UF'].isin(regiao_norte)]


IDEB varia de 0 a 10

SAEB vai de 0 a 500

Aprovação de 0 a 100

Preciso transformar String/Object em float e ajustar a base dos valores acima para que não fique desproporcional criando padronização

In [ ]:
colunas_numericas = [
    'VL_OBSERVADO_2005', 'VL_OBSERVADO_2023',
    'VL_APROVACAO_2005_1', 'VL_APROVACAO_2005_2', 'VL_APROVACAO_2005_3', 'VL_APROVACAO_2005_4',
    'VL_APROVACAO_2023_1', 'VL_APROVACAO_2023_2', 'VL_APROVACAO_2023_3', 'VL_APROVACAO_2023_4',
    'VL_NOTA_PORTUGUES_2005', 'VL_NOTA_PORTUGUES_2023',
    'VL_NOTA_MATEMATICA_2005', 'VL_NOTA_MATEMATICA_2023'
]
df[colunas_numericas] = df[colunas_numericas].apply(pd.to_numeric, errors='coerce')


agora a transformação de string e object em float

In [ ]:
# Cópias
df['IDEB_2005'] = df['VL_OBSERVADO_2005']
df['IDEB_2023'] = df['VL_OBSERVADO_2023']
df['SAEB_PORT_2005'] = df['VL_NOTA_PORTUGUES_2005']
df['SAEB_PORT_2023'] = df['VL_NOTA_PORTUGUES_2023']
df['SAEB_MAT_2005'] = df['VL_NOTA_MATEMATICA_2005']
df['SAEB_MAT_2023'] = df['VL_NOTA_MATEMATICA_2023']

# Variações e médias
df['VARIACAO_IDEB'] = df['IDEB_2023'] - df['IDEB_2005']
df['MEDIA_APROVACAO_2005'] = df[['VL_APROVACAO_2005_1', 'VL_APROVACAO_2005_2', 'VL_APROVACAO_2005_3', 'VL_APROVACAO_2005_4']].mean(axis=1)
df['MEDIA_APROVACAO_2023'] = df[['VL_APROVACAO_2023_1', 'VL_APROVACAO_2023_2', 'VL_APROVACAO_2023_3', 'VL_APROVACAO_2023_4']].mean(axis=1)
df['VARIACAO_APROVACAO'] = df['MEDIA_APROVACAO_2023'] - df['MEDIA_APROVACAO_2005']




limpeza estilo crisp

In [ ]:
df = df.drop_duplicates()


In [ ]:
colunas_essenciais = [
    'IDEB_2005', 'IDEB_2023', 'VARIACAO_IDEB',
    'MEDIA_APROVACAO_2005', 'MEDIA_APROVACAO_2023', 'VARIACAO_APROVACAO',
    'SAEB_PORT_2005', 'SAEB_PORT_2023', 'SAEB_MAT_2005', 'SAEB_MAT_2023'
]
df = df.dropna(subset=colunas_essenciais)

In [ ]:
from sklearn.preprocessing import StandardScaler

colunas_cluster = colunas_essenciais
X = df[colunas_cluster].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


visualização


In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

wcss = []
for k in range(1, 101):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 101), wcss, marker='o')
plt.title('Método do Cotovelo')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia (WCSS)')
plt.grid(True)
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)


In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

colunas_radar = [
    'IDEB_2023', 'VARIACAO_IDEB',
    'MEDIA_APROVACAO_2023', 'VARIACAO_APROVACAO',
    'SAEB_PORT_2023', 'SAEB_MAT_2023'
]

cores = ['#66c2a5', '#fc8d62', '#8da0cb']
nomes_personas = ['Persona A', 'Persona B', 'Persona C']

medias_por_cluster = df.groupby('cluster')[colunas_radar].mean()

scaler = MinMaxScaler()
medias_normalizadas = pd.DataFrame(
    scaler.fit_transform(medias_por_cluster),
    columns=colunas_radar,
    index=medias_por_cluster.index
)

labels = colunas_radar
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, axs = plt.subplots(1, len(medias_normalizadas), figsize=(5*len(medias_normalizadas), 5), subplot_kw=dict(polar=True))
quantidade_por_cluster = df['cluster'].value_counts().sort_index()

for i in range(len(medias_normalizadas)):
    valores = medias_normalizadas.iloc[i].tolist()
    valores += valores[:1]
    ax = axs[i] if len(medias_normalizadas) > 1 else axs
    ax.yaxis.set_visible(False)
    ax.plot(angles, valores, color=cores[i], linewidth=2)
    ax.fill(angles, valores, alpha=0.25, color=cores[i])
    ax.set_title(f'{nomes_personas[i]} - {quantidade_por_cluster[i]} registros', color=cores[i], size=13)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_yticklabels([])

plt.tight_layout()
plt.show()

# Salvar imagem
fig.savefig("radar_clusters.png", dpi=300)
fig.savefig("radar_clusters.pdf")


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

colunas_radar = [
    'IDEB_2023', 'VARIACAO_IDEB',
    'MEDIA_APROVACAO_2023', 'VARIACAO_APROVACAO',
    'SAEB_PORT_2023', 'SAEB_MAT_2023'
]

cores = ['#66c2a5', '#fc8d62', '#8da0cb', "#fc8d63"]
nomes_personas = ['Persona A', 'Persona B', 'Persona C', 'Persona D']

medias_por_cluster = df.groupby('cluster')[colunas_radar].mean()

scaler = MinMaxScaler()
medias_normalizadas = pd.DataFrame(
    scaler.fit_transform(medias_por_cluster),
    columns=colunas_radar,
    index=medias_por_cluster.index
)

labels = colunas_radar
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, axs = plt.subplots(1, len(medias_normalizadas), figsize=(5*len(medias_normalizadas), 5), subplot_kw=dict(polar=True))
quantidade_por_cluster = df['cluster'].value_counts().sort_index()

for i in range(len(medias_normalizadas)):
    valores = medias_normalizadas.iloc[i].tolist()
    valores += valores[:1]
    ax = axs[i] if len(medias_normalizadas) > 1 else axs
    ax.yaxis.set_visible(False)
    ax.plot(angles, valores, color=cores[i], linewidth=2)
    ax.fill(angles, valores, alpha=0.25, color=cores[i])
    ax.set_title(f'{nomes_personas[i]} - {quantidade_por_cluster[i]} registros', color=cores[i], size=13)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_yticklabels([])

plt.tight_layout()
plt.show()

# Salvar imagem
fig.savefig("radar_clusters.png", dpi=300)
fig.savefig("radar_clusters.pdf")


## Continuidade

Aproveitar documentação das variáveis, comparar escolhas de k com versões posteriores e preservar diferenças; substituir caminho planilhaIDEB.xlsx por entrada documentada.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.